In [1]:
using CMPSExcitations

In [2]:
# canonical basis
function projection_matrix(D, M)
    dim_in = D^2 + D      # input: E (DxD) + F (D)
    dim_out = 2 * D^2     # output: W1, W2 (both DxD)
    P = zeros(ComplexF64, dim_out, dim_in)

    for j in 1:dim_in
        e = zeros(Float64, dim_in)
        e[j] = 1.0

        E = reshape(view(e, 1:D^2), D, D)
        F = view(e, D^2+1:dim_in)

        W1 = E
        W2 = E + M * Diagonal(F) / M

        P[:, j] = vcat(vec(W1), vec(W2))
    end

    return Matrix(qr(P).Q) # moves to an orthogonal projection
end

# canonical basis
function excitation_matrix(Heff, D)
    dim = 2 * D^2
    M = zeros(ComplexF64, dim, dim)

    for j in 1:dim
        e = zeros(ComplexF64, dim)
        e[j] = 1.0
        W1 = reshape(view(e, 1:D^2), D, D)
        W2 = reshape(view(e, D^2+1:dim), D, D)
        W1p, W2p = Heff((Constant(W1), Constant(W2)))
        M[:, j] = vcat(vec(W1p[]), vec(W2p[]))
    end

    return M
end

function excitation_matrix_constrained(Heff, M)
    D = size(M, 1) # R = MDᵣ/M
    H = excitation_matrix(Heff, D)
    P = projection_matrix(D, M)

    P' * H * P, P
end

excitation_matrix_constrained (generic function with 1 method)

In [3]:
Hsingle(c, μ) = ∫(2 * ∂ψ̂' * ∂ψ̂ - 2 * μ * ψ̂' * ψ̂ + 4 * c * (ψ̂')^2 * ψ̂^2, (-Inf, +Inf));
Hcoupled(c, μ) = ∫(
    (∂ψ̂₁' * ∂ψ̂₁ - μ * ψ̂₁' * ψ̂₁ + c * (ψ̂₁')^2 * ψ̂₁^2 +
     ∂ψ̂₂' * ∂ψ̂₂ - μ * ψ̂₂' * ψ̂₂ + c * (ψ̂₂')^2 * ψ̂₂^2 +
     2 * c * (ψ̂₁') * (ψ̂₂') * ψ̂₂ * ψ̂₁), (-Inf, +Inf));

In [5]:
Hsingle_ll(c, μ) = ∫(∂ψ̂' * ∂ψ̂ - μ * ψ̂' * ψ̂ + c * (ψ̂')^2 * ψ̂^2, (-Inf, +Inf));
c, μ = 10., 5.
tol = 1e-10
Ds = [4, 8]
D = maximum(Ds)

HLL = Hsingle(c, μ)
stateLL = find_groundstate(Ds, HLL, InfiniteCMPS, optalg=LBFGS(80; verbosity=1, maxiter=7000, gradtol=tol), gradtol=tol)
println("Energy density: ", expval(HLL.h, stateLL)[], "\n Particle density: ", expval(ψ̂' * ψ̂, stateLL)[], "\n Order parameter: ", expval(ψ̂, stateLL)[])

HCLL = Hcoupled(c, μ)
stateCLL = InfiniteCMPS(stateLL.Q, (stateLL.Rs[1], stateLL.Rs[1]));
leftgauge!(stateCLL)
println("Energy density: ", expval(HCLL.h, stateCLL)[], "\n Particle density: ", expval(ψ̂' * ψ̂, stateCLL)[], "\n Order parameter: ", expval(ψ̂, stateCLL)[])

Optimizing D=4


┌ Info: UniformCMPS ground state: initialization with e = 1059.603055390351
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/infinitecmps/groundstate.jl:115
┌ Info: LBFGS: converged after 206 iterations: f = -5.050019379406, ‖∇f‖ = 4.8121e-11
└ @ OptimKit /home/ashankar/.julia/packages/OptimKit/xpmbV/src/lbfgs.jl:138


D = 4 | InfiniteCMPS{Constant{Matrix{Float64}}, 1}
  0.174216 seconds (947.38 k allocations: 42.106 MiB, 3.85% gc time)
---------------
Optimizing D=8


┌ Info: UniformCMPS ground state: converged after 207 iterations: e = -5.050019379406, ‖∇e‖ = 4.8121e-11
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/infinitecmps/groundstate.jl:127
┌ Info: UniformCMPS ground state: initialization with e = -5.050019379954
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/infinitecmps/groundstate.jl:115
┌ Info: LBFGS: converged after 428 iterations: f = -5.101997460806, ‖∇f‖ = 1.8680e-11
└ @ OptimKit /home/ashankar/.julia/packages/OptimKit/xpmbV/src/lbfgs.jl:138


D = 8 | InfiniteCMPS{Constant{Matrix{Float64}}, 1}
  3.773016 seconds (4.18 M allocations: 378.974 MiB, 1.02% gc time)
---------------
Energy density: -5.101997460806262
 Particle density: 0.785006418777062
 Order parameter: 0.4309827953238506
Energy density: -2.6552416754694725

┌ Info: UniformCMPS ground state: converged after 429 iterations: e = -5.101997460806, ‖∇e‖ = 1.8680e-11
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/infinitecmps/groundstate.jl:127



 Particle density: 0.40133751372569504
 Order parameter: 0.3092564728778627


In [6]:
function leftgaugetv(Q, R, V, W1, W2, p)
    # assuming Q, R are in cMPS left gauge
    X, _ = linsolve(-R' * (W1 + W2) - V) do X
        (Q * X - X * Q) + 1im * p * X + 2 * R' * (R * X - X * R)
    end

    return V + (Q * X - X * Q) + 1im * p * X, W1 + (R * X - X * R), W2 + (R * X - X * R)
end

leftgaugetv (generic function with 1 method)

In [23]:
p = 1
Q, R = stateCLL.Q[], stateCLL.Rs[1][];
ρR = rightenv(stateCLL)[1][];
W1, W2 = R, -R;

V, W1, W2 = leftgaugetv(Q, R, zero(Q), R, -R, p)

(ComplexF64[0.0 + 0.0im 0.0 + 0.0im … 0.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 0.0 + 0.0im … 0.0 + 0.0im 0.0 + 0.0im; … ; 0.0 + 0.0im 0.0 + 0.0im … 0.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 0.0 + 0.0im … 0.0 + 0.0im 0.0 + 0.0im], ComplexF64[0.6083335782447142 + 0.0im -0.5550491263747226 + 0.0im … -0.7696735629242679 + 0.0im 0.0638415843749876 + 0.0im; -0.038741030051683036 + 0.0im -0.5585186363422108 + 0.0im … 0.6143401649799389 + 0.0im -0.9154281162428514 + 0.0im; … ; -0.06383418775017022 + 0.0im -0.017616432537905596 + 0.0im … -0.24557479152347372 + 0.0im -0.032466854391947304 + 0.0im; 0.09464987298289425 + 0.0im 0.0432977373288882 + 0.0im … 0.34568056321619584 + 0.0im 0.4200594830657917 + 0.0im], ComplexF64[-0.6083335782447142 + 0.0im 0.5550491263747226 + 0.0im … 0.7696735629242679 + 0.0im -0.0638415843749876 + 0.0im; 0.038741030051683036 + 0.0im 0.5585186363422108 + 0.0im … -0.6143401649799389 + 0.0im 0.9154281162428514 + 0.0im; … ; 0.06383418775017022 + 0.0im 0.017616432537905596 + 0.0im …

In [13]:
space = InfiniteCMPSExcitationSpace(p, stateCLL, stateCLL)
Heff = excitation_operator(HCLL, space);

In [16]:
HW1, HW2 = getindex.(Heff((Constant(W1), Constant(W2))));

In [26]:
tr(HW1 * ρR * W1' + HW2 + ρR * W2')

48.14640234437253 - 8.269569214697945im